# 01 — Conhecendo e organizando a base

Antes de prever uma série temporal, precisamos organizar os dados no tempo.

Neste notebook, vamos:

- carregar a base de bicicletas;
- identificar a coluna de tempo;
- identificar a variável de demanda;
- converter a data;
- ordenar os registros;
- verificar se existem valores ausentes.

## Imports e caminhos

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

# raiz do projeto
if Path.cwd().name == "notebooks":
    raiz_projeto = Path.cwd().parent
else:
    raiz_projeto = Path.cwd()

sys.path.append(str(raiz_projeto / "src"))

from visual_utils import grafico_linha_padrao

caminho_base = raiz_projeto / "data" / "bike.csv"
caminho_saida = raiz_projeto / "outputs" / "tabelas" / "base_bike_organizada.csv"

print("Base:", caminho_base)

## Carregar a base

In [ ]:
df = pd.read_csv(caminho_base)

print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

df.head()

## Olhar tipos e colunas

In [ ]:
df.info()

## Renomear as colunas principais

In [ ]:
df = df.rename(columns={
    "datetime": "data_hora",
    "count": "demanda",
    "temp": "temperatura",
    "humidity": "umidade",
    "windspeed": "velocidade_vento",
    "season": "estacao",
    "holiday": "feriado",
    "workingday": "dia_util",
    "weather": "clima",
    "casual": "usuarios_casuais",
    "registered": "usuarios_registrados",
    "atemp": "sensacao_termica"
})

df[["data_hora", "demanda"]].head()

## Converter data e ordenar

In [ ]:
df["data_hora"] = pd.to_datetime(df["data_hora"])

df = df.sort_values("data_hora").reset_index(drop=True)

print("Primeira data:", df["data_hora"].min())
print("Última data:", df["data_hora"].max())

df[["data_hora", "demanda"]].head()

## Criar variáveis de calendário

Agora que a coluna `data_hora` está no formato correto, podemos extrair informações do calendário.

Essas variáveis ajudam a investigar padrões por hora, mês e dia da semana nos próximos notebooks.

In [ ]:
df["ano"] = df["data_hora"].dt.year
df["mes"] = df["data_hora"].dt.month
df["dia"] = df["data_hora"].dt.day
df["hora"] = df["data_hora"].dt.hour
df["dia_semana"] = df["data_hora"].dt.dayofweek

mapa_dias_semana = {
    0: "segunda-feira",
    1: "terça-feira",
    2: "quarta-feira",
    3: "quinta-feira",
    4: "sexta-feira",
    5: "sábado",
    6: "domingo"
}

df["nome_dia_semana"] = df["dia_semana"].map(mapa_dias_semana)

df[["data_hora", "demanda", "ano", "mes", "dia", "hora", "nome_dia_semana"]].head()

## Criar índice temporal

In [ ]:
df_temporal = df.set_index("data_hora")

df_temporal.head()

## Checar frequência e ausentes

In [ ]:
print("Frequência mais comum:")
print(df_temporal.index.to_series().diff().value_counts().head())

print("\nValores ausentes:")
print(df_temporal.isna().sum())

## Primeira visualização rápida

In [ ]:
df_grafico = df_temporal.reset_index()

fig = grafico_linha_padrao(
    df=df_grafico,
    x="data_hora",
    y="demanda",
    titulo="Demanda de bicicletas ao longo do tempo",
    labels={
        "data_hora": "Data e hora",
        "demanda": "Demanda"
    },
    altura=500
)

fig.show()

## Salvar base organizada

In [ ]:
df_temporal.reset_index().to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

print("Base organizada salva em:")
print(caminho_saida)